In [1]:
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
import pandas as pd

df=pd.read_csv("clean_data.csv")

X = df['Narrative Text']
y = df['Event Label']

X_train, X_temp, y_train, y_temp = train_test_split(
    X, y, test_size=0.30, random_state=42, stratify=y
)
X_dev, X_test, y_dev, y_test = train_test_split(
    X_temp, y_temp, test_size=0.50, random_state=42, stratify=y_temp
)

In [2]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
import joblib

lr_model= Pipeline([
    ("tfidf",TfidfVectorizer(stop_words='english', max_features=10000)),
    ("clf",LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42))
])

lr_model.fit(X_train,y_train)
# Serialization
pipeline_lr="lr_model_pipeline.joblib"
joblib.dump(lr_model,pipeline_lr,compress=3)

# Deserialization
lr_load=joblib.load(pipeline_lr)

y_pred=lr_load.predict(X_test)

In [4]:
from sklearn.metrics import classification_report
import mlflow

print(classification_report(y_test, y_pred))

report=classification_report(y_test, y_pred,output_dict=True)

mlflow.set_experiment(experiment_name="aviation-classification")
with mlflow.start_run(run_name="Logistic Regression"):
     mlflow.log_param("model_type","logistic_regression")
     mlflow.log_param("max_features",10000)
     mlflow.log_param("stop_words","english")
     mlflow.log_param("random_state",42)
     mlflow.log_param("class_weighting","balanced")

     mlflow.log_metric("test_accuracy",report["accuracy"])
     mlflow.log_metric("macro_f1",report["macro avg"]["f1-score"])
     mlflow.log_metric("weighted_f1",report["weighted avg"]["f1-score"])
     mlflow.log_metric("ground_event_f1",report["Ground Event"]["f1-score"])
     mlflow.log_metric("ground_event_recall",report["Ground Event"]["recall"])
     

                   precision    recall  f1-score   support

        ATC Issue       0.64      0.69      0.66       275
         Conflict       0.57      0.65      0.61       123
        Deviation       0.74      0.67      0.70       589
Equipment Problem       0.94      0.88      0.91      1301
     Ground Event       0.23      0.54      0.32        35
   Inflight Event       0.44      0.72      0.55        61

         accuracy                           0.79      2384
        macro avg       0.59      0.69      0.63      2384
     weighted avg       0.81      0.79      0.80      2384

